In [2]:
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import torch
from data import load
import pandas as pd
import re
import optuna
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter


In [22]:
storage_name = "sqlite:///db/optuna_study.db"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
EPOCHS = 10
out_path = "models/model.pt"

cpu


In [3]:
print(device)  # torch.xpu is the API for Intel GPU support

cpu


In [4]:
dataset_1 = load("realDonaldTrump_bf_office.csv")
dataset_2 = load("realDonaldTrump_in_office.csv")
dataset = pd.concat([dataset_1, dataset_2], ignore_index=True)
dataset.columns = dataset.columns.str.strip()

In [5]:
dataset = dataset['Tweet Text'].to_frame()
print(dataset.shape)
print(dataset.columns)

(32352, 1)
Index(['Tweet Text'], dtype='object')


In [6]:
def clean(text):
     # Remove URL
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'["\'\`""\'\'"""''«»„‟]', '', text)
    # Solo mantener: letras, números, espacios, punto y coma
    text = re.sub(r'[^a-zA-Z0-9\s\.\,]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


In [7]:
def text(dataset):
    text = ' '.join(dataset['Tweet Text'].tolist())
    return text


In [8]:
dataset['Tweet Text'] = dataset['Tweet Text'].apply(clean)
text = text(dataset)


In [9]:
print(len(text))

3083591


In [9]:
def tokenize(text ):
   characters = set(text)
   vocab_size = len(characters)
   chart_idx = {c : i for i, c in enumerate(characters)}
   idx_char = {i : c for i, c in enumerate(characters)}
   return characters , vocab_size,chart_idx ,idx_char


In [10]:
characters , vocab_size,chart_idx ,idx_char = tokenize(text)

In [11]:
def create_sequences(text, chart_idx, seq_length=100):
    sequences = []
    targets = []
    for i in range(len(text) - seq_length):
        seq = text[i:i+seq_length]
        target = text[i+seq_length]

        #Convert to indxes
        seq_encoded = [chart_idx[c] for c in seq]
        target_encoded = chart_idx[target]

        sequences.append(seq_encoded)
        targets.append(target_encoded)

    sequences = np.array(sequences)
    targets = np.array(targets)
    return sequences , targets

In [12]:
sequences , target = create_sequences(text, chart_idx)
print(sequences.shape)
print(target.shape)

(3083491, 100)
(3083491,)


In [13]:
def create_dataset(sequences, target, batch_size, train_split=0.8):
    """Create train and validation dataloaders"""
    X = torch.tensor(sequences, dtype=torch.long)
    y = torch.tensor(target, dtype=torch.long)

    # Split into train and validation
    split_idx = int(len(X) * train_split)
    X_train, X_val = X[:split_idx], X[split_idx:]
    y_train, y_val = y[:split_idx], y[split_idx:]

    # Create dataloaders
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

In [14]:
class LSTMModel(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers , dropout , bidirectional):
        super(LSTMModel, self).__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim)
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True , dropout=dropout , bidirectional=bidirectional)
        # Handle bidirectional: output size doubles if bidirectional=True
        output_size = hidden_dim * (2 if bidirectional else 1)
        self.fc = torch.nn.Linear(output_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x.long())
        out, _ = self.lstm(x)
        # Take last output
        lstm_out = out[:, -1, :]
        output = self.fc(lstm_out)
        return output

In [15]:
def train(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    num_batches = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # Forward pass
        output = model(X_batch)
        loss = criterion(output, y_batch)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches


In [16]:
def validate_lstm(model, val_loader, criterion, device):
    """Validate the model and compute perplexity"""
    model.eval()
    total_loss = 0
    num_batches = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            output = model(X_batch)
            loss = criterion(output, y_batch)

            total_loss += loss.item()
            num_batches += 1

    avg_loss = total_loss / num_batches
    perplexity = np.exp(avg_loss)

    return avg_loss, perplexity


In [17]:
def character_accuracy(model, val_loader, device):
    """Calculate character-level accuracy"""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            output = model(X_batch)
            predictions = torch.argmax(output, dim=1)

            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    accuracy = correct / total if total > 0 else 0
    return accuracy

In [19]:
def objective(trial , sequences , target):

    embedding_dim = trial.suggest_int("embedding_dim", 50, 200)
    hidden_dim = trial.suggest_int("hidden_dim", 50, 200)
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    bidirectional = trial.suggest_categorical("bidirectional", [True, False])
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)

    model = LSTMModel(vocab_size, embedding_dim, hidden_dim, num_layers , dropout , bidirectional).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(params= model.parameters(),lr=learning_rate)

    # Create train and val loaders
    train_loader, val_loader = create_dataset(sequences, target, batch_size)

    for epoch in range(3):
         train(model, train_loader, criterion, optimizer, device)

    # Return validation loss
    val_loss, _ = validate_lstm(model, val_loader, criterion, device)
    return val_loss

In [ ]:
study = optuna.create_study(
    study_name="study4",
    storage=storage_name,
    direction="maximize",
    load_if_exists=True
)

study.optimize(lambda trial: objective(trial, sequences[:500000], target[:500000]), n_trials=8)

[I 2026-05-14 21:39:39,987] Using an existing study with name 'study4' instead of creating a new one.
C:\Users\beni7\PycharmProjects\chrome-dino\.env\lib\site-packages\torch\nn\modules\rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.12018567344099956 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
C:\Users\beni7\PycharmProjects\chrome-dino\.env\lib\site-packages\torch\autograd\graph.py:869: UserWarning: The detected GPU (Intel(R) Iris(R) Xe Graphics) is not officially supported by PyTorch XPU. Running workloads on this device may result in unexpected behavior.
For stable and fully supported execution, please use GPUs based on Intel Arc (Alchemist) series or newer.
Refer to the hardware prerequisites for more information: https://github.com/pytorch/pytorch/blob/main/docs/source/notes/get_start_xpu.rst#hardware-prerequisite (Triggered internally at C:\actions-ru

In [24]:
def save_checkpoint(model, best_params, checkpoint_path="model.pt"):

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "best_params": best_params,
    }

    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved to {checkpoint_path}")

In [25]:
writer = SummaryWriter('runs')

In [ ]:
study = optuna.load_study(study_name="study4", storage=storage_name)

embedding_dim = study.best_params["embedding_dim"]
hidden_dim = study.best_params["hidden_dim"]
num_layers = study.best_params["num_layers"]
dropout = study.best_params["dropout"]
bidirectional = study.best_params["bidirectional"]
batch_size = study.best_params["batch_size"]
learning_rate = study.best_params["learning_rate"]

model = LSTMModel(vocab_size, embedding_dim, hidden_dim, num_layers , dropout , bidirectional).to(device)
loss_fn=nn.CrossEntropyLoss()


optimizer = torch.optim.Adam(
            params= model.parameters(),
            lr=learning_rate)

# Create train and validation loaders
train_loader, val_loader = create_dataset(sequences, target, batch_size)

early_stopping = False
best_val_loss = np.inf


print("Training with best hyperparameters...")
print(f"Embedding: {embedding_dim}, Hidden: {hidden_dim}, Layers: {num_layers}")
print(f"Learning Rate: {learning_rate}, Batch Size: {batch_size}\n")

#Training Process with Validation
for epoch in range(EPOCHS):
        # Train
        train_loss = train(model, train_loader, loss_fn, optimizer, device)
        # Validate
        val_loss, perplexity = validate_lstm(model, val_loader, loss_fn,device)
        # Compute accuracy
        char_acc = character_accuracy(model, val_loader, device)

        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('Metrics/perplexity', perplexity, epoch)
        writer.add_scalar('Metrics/char_accuracy', char_acc, epoch)

        print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}, Perplexity: {perplexity:.2f}, Char Acc: {char_acc:.2%}")
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            early_stopping = True
            # Save best model
            save_checkpoint(model, study.best_params, checkpoint_path=out_path)


writer.close()
print(f"\nTraining completed!")

if not early_stopping:
    save_checkpoint(model, study.best_params, checkpoint_path=out_path)

print(f"Saved model {out_path}")


Training with best hyperparameters...
Embedding: 161, Hidden: 124, Layers: 2
Learning Rate: 0.008965759972624898, Batch Size: 64



C:\Users\beni7\PycharmProjects\chrome-dino\.env\lib\site-packages\torch\autograd\graph.py:869: UserWarning: The detected GPU (Intel(R) Iris(R) Xe Graphics) is not officially supported by PyTorch XPU. Running workloads on this device may result in unexpected behavior.
For stable and fully supported execution, please use GPUs based on Intel Arc (Alchemist) series or newer.
Refer to the hardware prerequisites for more information: https://github.com/pytorch/pytorch/blob/main/docs/source/notes/get_start_xpu.rst#hardware-prerequisite (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\xpu\XPUFunctions.cpp:134.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1/10 - Train Loss: 1.788324, Val Loss: 1.833109, Perplexity: 6.25, Char Acc: 45.84%
Checkpoint saved to models/model.pt


In [1]:
%load_ext tensorboard
%tensorboard --logdir runs

In [19]:

def generate_text(model, start_text, length=500, temperature=0.8):

    model.eval()

    # Convert start_text to indices
    current_seq = [chart_idx[c] for c in start_text]
    generated_text = start_text

    with torch.no_grad():
        for _ in range(length):
            # Take last 100 characters (seq_length)
            input_seq = torch.tensor(current_seq[-100:], dtype=torch.long).unsqueeze(0).to(device)

            # Predict next character
            output = model(input_seq)

            # Apply temperature
            logits = output[0] / temperature
            probs = torch.softmax(logits, dim=0)

            # Sample character (not always take maximum)
            next_char_idx = torch.multinomial(probs, 1).item()
            next_char = idx_char[next_char_idx]

            generated_text += next_char
            current_seq.append(next_char_idx)

    return generated_text


In [25]:
# Load best model for generation
checkpoint = torch.load(out_path, map_location=device)

best = checkpoint["best_params"]
embedding_dim = best["embedding_dim"]
hidden_dim = best["hidden_dim"]
num_layers = best["num_layers"]
dropout = best["dropout"]
bidirectional = best["bidirectional"]

model =  LSTMModel(vocab_size, embedding_dim, hidden_dim, num_layers , dropout , bidirectional).to(device)
model.load_state_dict(checkpoint["model_state_dict"])


<All keys matched successfully>

In [26]:
# ===== GENERATE SPEECHES =====
prompts = [
    "the best deals",
    "i have built",
    "america is great",
    "my experience"
]


print("GENERATING TRUMP-STYLE SPEECHES")


for prompt in prompts:
    print(f"\nPROMPT: {prompt}")
    print("-" * 80)

    speech = generate_text(model, prompt, length=500, temperature=0.85)
    print(speech)
    print()



GENERATING TRUMP-STYLE SPEECHES

PROMPT: the best deals
--------------------------------------------------------------------------------
the best dealss9 vl2swo z445 .h9ulu zw.hlw.l.whqzl .xlxgl5g4lwlq n9lx9cg.xl9p9uwv9.9lfwzqlzq .yl5g4lwlv4sw9 .wzguylq9lwcl 00u9.zwo9lu4.lkwhlw.loq c0gu09uwv9lzq9loq4h9ulw.lkwhu9vl. zw.hlxwu.h4 lsw29lzq .yl5g4lf vlc y9 cc5z9uwks9c9u9vzu wguzg.9lzq9lswy9l k9uwz9xlvz .xlkuwola4vzlo .l.5 .lk9lq z9xlkwsswg.lzglx9 smlhggkw slkwsswg.vlzq zlx9kglq9losw.zg.lu90gz4vdw lk9s5lzglc yw.hlkugflfqwz9mlzu4c0lzq9lx9cgou0 z9.vlwzlogsslou9 zlf zqlhgsck9l2gul l6 x lwvlv99lvglhu9 zl4.o9lsgn9lgqlzu4c01bi3lvw.o9lzq9lqwhqzl 00ugvzc ..


PROMPT: i have built
--------------------------------------------------------------------------------
i have builtlzq9l 00u9.zwo9mlvzugc9l2gulg4ulq9s09xlg2lfq zl uuwg.l .xlkgux9uvlwvl4. .xsgksglu4c09lwvqg4zlswy9lkgzqlxg sl.gzl0ugks9clg2log.zwccl kwssg.wz9ul4vmlu .wz5l ..lw.l.99x9ul .xmlk9lu9 sxg. sxzu4c0l .xlvqgc.lgsxu9c9.zl2gulzq9lhgu.wvl0u x